In [ ]:
#CELDA 0
import subprocess, sys

# Verifica dependencias
subprocess.run([sys.executable, "-m", "pip", "install", 
                "python-dotenv", "openai", "-q"])

from dotenv import load_dotenv
import os
load_dotenv()

token = os.getenv("GITHUB_TOKEN")
url   = os.getenv("OPENAI_BASE_URL")
print(f"Token:   {'✅ OK' if token else '❌ Renovar en github.com/settings/tokens'}")
print(f"API URL: {'✅ ' + url if url else '❌ Falta en .env'}")

In [ ]:

#CELDA 2
import os
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_API_KEY"] = "dummy"

from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Configuración del modelo (igual al chat con ia.)
token = os.environ["GITHUB_TOKEN"]
endpoint = "https://models.github.ai/inference"
model_name = "openai/gpt-4.1"

llm = ChatOpenAI(
    base_url=endpoint,
    api_key=token,
    model=model_name,
    temperature=0.1,
    streaming=True
)

print("✓ Modelo listo")

In [ ]:
#CELDA 2
# Base de conocimiento - Vulcanizadora
documents = [
    "El parche de neumático es una reparación que se aplica desde el interior de la llanta para sellar pinchazos. Se recomienda para cortes menores a 6mm en la banda de rodamiento.",
    "El cambio de neumático consiste en retirar la llanta dañada y montar una nueva. Se recomienda cambiar los neumáticos cuando el indicador de desgaste es visible o tienen más de 5 años.",
    "La alineación de ruedas ajusta los ángulos de las ruedas según las especificaciones del fabricante. Una mala alineación causa desgaste irregular y problemas de dirección.",
    "El balanceo de ruedas distribuye el peso uniformemente alrededor del eje. Se recomienda cada 10.000 km o cuando se siente vibración en el volante.",
    "El inflado correcto de neumáticos es clave para la seguridad. La presión recomendada suele estar entre 30 y 35 PSI según el vehículo, y debe revisarse mensualmente.",
    "Los neumáticos all-season son aptos para uso general. Los de alto rendimiento ofrecen mejor agarre pero mayor desgaste. Los off-road están diseñados para terrenos irregulares.",
    "Señales de que un neumático necesita reemplazo: profundidad del dibujo menor a 1.6mm, grietas en los laterales, bultos o deformaciones visibles.",
    "El nitrógeno en neumáticos mantiene la presión más estable que el aire normal. Se recomienda especialmente en vehículos de carga o flotas.",
]

print(f"📚 Base de conocimiento cargada con {len(documents)} documentos")

In [ ]:
#CELDA 3

# Función de recuperación por palabras clave
def simple_retrieval(query, documents):
    relevant_docs = []
    query_lower = query.lower()
    
    for doc in documents:
        if any(word in doc.lower() for word in query_lower.split()):
            relevant_docs.append(doc)
    
    return relevant_docs[:3]

print("✅ Función de recuperación lista")

In [ ]:
#CELDA 4

# Función de generación usando el llm que ya tienes configurado
from langchain_core.messages import HumanMessage

def generate_response(llm, query, context):
    prompt = f"""Eres un asistente experto de una vulcanizadora.
Responde de forma clara, amable y profesional basándote ÚNICAMENTE en el contexto proporcionado.
Si la información no está en el contexto, indícalo amablemente.

Contexto:
{context}

Pregunta del cliente: {query}

Respuesta:"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

print("✅ Función de generación lista")

In [ ]:
# CELDA 5
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

llm_rag = ChatOpenAI(
    base_url="https://models.github.ai/inference",
    api_key=token,
    model="openai/gpt-4.1",
    temperature=0.1,
    streaming=False
)

# Verificar que llm_rag funciona
test = llm_rag.invoke([HumanMessage(content="responde solo: ok")])
print("✅ llm_rag funciona:", test.content)

def generate_response(model, query, context):
    prompt = f"""Eres un asistente experto de una vulcanizadora.
Responde de forma clara, amable y profesional basándote ÚNICAMENTE en el contexto proporcionado.
Si la información no está en el contexto, indícalo amablemente.

Contexto:
{context}

Pregunta del cliente: {query}

Respuesta:"""

    response = model.invoke([HumanMessage(content=prompt)])
    return response.content

print("✅ Función de generación lista")

In [ ]:
#CELDA 6

# Celda 5 - Prueba del sistema RAG completo
query = "¿Cuándo debo cambiar mis neumáticos?"

print(f"❓ Pregunta: {query}")
print("="*50)

# Paso 1: Recuperación
relevant_docs = simple_retrieval(query, documents)
print(f"📋 Documentos relevantes encontrados: {len(relevant_docs)}")
for i, doc in enumerate(relevant_docs, 1):
    print(f"  {i}. {doc[:80]}...")

# Paso 2: Generación
if relevant_docs:
    context = "\n".join(relevant_docs)
    response = generate_response(llm_rag, query, context)
    print(f"\n🤖 Respuesta del asistente:")
    print(response)
else:
    print("❌ No se encontró información relevante.")